# Feature Engineering — NorthStar

Produces the 10 features from Section 2.4. Three layers: row-level features on deliveries, cross-file linked features (incidents and complaints), and per-entity rollups for drivers and vehicles.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR     = Path('/content/drive/MyDrive/northstar-databases-analytics')
CLEANED_DIR  = BASE_DIR / 'data' / 'cleaned'
FEATURES_DIR = BASE_DIR / 'data' / 'features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

customers  = pd.read_csv(CLEANED_DIR / 'customers_cleaned.csv',  parse_dates=['signup_date'])
orders     = pd.read_csv(CLEANED_DIR / 'orders_cleaned.csv',     parse_dates=['order_created_at'])
deliveries = pd.read_csv(CLEANED_DIR / 'deliveries_cleaned.csv', parse_dates=['dispatch_time', 'delivery_completed_at'])
drivers    = pd.read_csv(CLEANED_DIR / 'drivers_cleaned.csv')
vehicles   = pd.read_csv(CLEANED_DIR / 'vehicles_cleaned.csv',   parse_dates=['commission_date'])
complaints = pd.read_csv(CLEANED_DIR / 'complaints_cleaned.csv', parse_dates=['created_at'])
incidents  = pd.read_csv(CLEANED_DIR / 'incidents_cleaned.csv',  parse_dates=['reported_at'])

print('Loaded.')

## 2. Row-level delivery features

In [ ]:
order_fields = orders[['order_id', 'promised_window_hours', 'pickup_zone', 'dropoff_zone']]
deliveries = deliveries.merge(order_fields, on='order_id', how='left')

deliveries['delivery_duration_minutes'] = (
    (deliveries['delivery_completed_at'] - deliveries['dispatch_time']).dt.total_seconds() / 60
)

deliveries['is_late'] = (
    deliveries['delivery_duration_minutes'] > deliveries['promised_window_hours'] * 60
).astype('boolean')
deliveries.loc[deliveries['delivery_duration_minutes'].isna(), 'is_late'] = pd.NA

deliveries['is_failed']     = (deliveries['delivery_status'] == 'Failed').astype(int)
deliveries['is_cross_zone'] = (deliveries['pickup_zone'] != deliveries['dropoff_zone']).astype(int)
deliveries['dispatch_hour'] = deliveries['dispatch_time'].dt.hour
deliveries['is_weekend']    = (deliveries['dispatch_time'].dt.dayofweek >= 5).astype(int)
deliveries['manual_override_intensity'] = (
    deliveries['manual_route_override_count'] / deliveries['route_distance_km']
).replace([np.inf, -np.inf], np.nan)

print('Row-level features added.')

## 3. Cross-file features

Join deliveries with incidents (by delivery_id) and complaints (by order_id) so each delivery row knows whether it had downstream issues.

In [ ]:
incident_keys = incidents[['delivery_id']].drop_duplicates()
incident_keys['had_incident'] = 1
deliveries = deliveries.merge(incident_keys, on='delivery_id', how='left')
deliveries['had_incident'] = deliveries['had_incident'].fillna(0).astype(int)

complaint_keys = complaints[['order_id']].drop_duplicates()
complaint_keys['had_complaint'] = 1
deliveries = deliveries.merge(complaint_keys, on='order_id', how='left')
deliveries['had_complaint'] = deliveries['had_complaint'].fillna(0).astype(int)

print('deliveries with incidents :', deliveries['had_incident'].sum())
print('deliveries with complaints:', deliveries['had_complaint'].sum())

deliveries.to_csv(FEATURES_DIR / 'deliveries_features.csv', index=False)

## 4. Driver rollup

In [ ]:
driver_features = (
    deliveries
    .groupby('driver_id', as_index=False)
    .agg(
        total_deliveries        = ('delivery_id', 'count'),
        failure_rate            = ('is_failed', 'mean'),
        late_rate               = ('is_late', lambda s: s.dropna().mean()),
        incidents_per_100       = ('had_incident', lambda s: s.mean() * 100),
        complaints_per_100      = ('had_complaint', lambda s: s.mean() * 100),
        mean_override_intensity = ('manual_override_intensity', 'mean'),
        mean_rating_received    = ('customer_rating_post_delivery', 'mean'),
    )
)

driver_features = driver_features.merge(
    drivers[['driver_id', 'base_zone', 'employment_type', 'years_experience',
             'training_score', 'driver_rating', 'active_flag']],
    on='driver_id', how='right'
)

driver_features.to_csv(FEATURES_DIR / 'driver_features.csv', index=False)
print(f'driver_features: {len(driver_features)} rows')

## 5. Vehicle rollup

In [ ]:
REF_DATE = pd.Timestamp('2026-01-01')

vehicle_features = vehicles.copy()
vehicle_features['vehicle_age_days'] = (REF_DATE - vehicle_features['commission_date']).dt.days
vehicle_features['km_per_day_in_service'] = np.where(
    vehicle_features['vehicle_age_days'] > 0,
    vehicle_features['odometer_km'] / vehicle_features['vehicle_age_days'],
    np.nan
)

vehicle_rollup = (
    deliveries
    .groupby('vehicle_id', as_index=False)
    .agg(
        deliveries_assigned = ('delivery_id', 'count'),
        incidents_per_100   = ('had_incident', lambda s: s.mean() * 100),
        failure_rate        = ('is_failed', 'mean'),
    )
)
vehicle_features = vehicle_features.merge(vehicle_rollup, on='vehicle_id', how='left')
vehicle_features.to_csv(FEATURES_DIR / 'vehicle_features.csv', index=False)
print(f'vehicle_features: {len(vehicle_features)} rows')

## 6. Output files

In [ ]:
for p in sorted(FEATURES_DIR.glob('*.csv')):
    df = pd.read_csv(p)
    print(f'{p.name}: {len(df)} rows, {len(df.columns)} cols')